In [10]:
import pandas as pd
import pydeck as pdk
import os
import streamlit as st
import pickle
from pydeck.types import String

In [11]:

#mapbox_token="pk.eyJ1IjoibWFwYm94cnMyMSIsImEiOiJjamdkdTU1MTIwMTM2Mnhxa3Y3ZXZ3eGt3In0.PtflK7MObAbmwY1E__H7Fg"

In [37]:
os.environ["OPENAI_API_KEY"] = st.secrets["OPENAI_API_KEY"]
os.environ["MAPBOX_TOKEN"] = st.secrets["MAPBOX_TOKEN"]
MAPBOX_TOKEN = st.secrets["MAPBOX_TOKEN"]

In [38]:
def load_centroids_asat():
    #dg = pd.read_csv("penguins.csv", engine="pyarrow")
  #  df = pd.read_json(df.to_json())
    dg = pd.read_pickle('updatejammingcentroids2d.pkl.gz')
    #return dg
    return dg[dg.cluster != -1]

def load_dftriple_asat():
    dg = pd.read_pickle('updatejammingdftriple2d.pkl.gz')
    return dg


def load_dfinfo_asat():
    dg = pd.read_pickle('updatejammingdfinfo2d.pkl.gz')
    #return dg
    return dg[dg['cluster'] != -1]

def load_source_dict():
    with open("updatesource_page_dict.pkl", "rb") as f:
        source_dict = pickle.load(f)
    return source_dict

def load_affil_geo_dict():
    with open("updateaffil_geo_dict.pkl", "rb") as f:
        affil_geo_dict = pickle.load(f)
    return affil_geo_dict

In [39]:
centroids = load_centroids_asat()
dftriple = load_dftriple_asat()
dfinfo = load_dfinfo_asat()
dfinfo['cluster_'] = dfinfo["cluster"].apply(str)
#dfgeo = load_dfgeo_asat()
source_dict = load_source_dict()
affil_geo_dict = load_affil_geo_dict()

kw_dict = dfinfo['keywords'].to_dict()

In [107]:
def get_affils_cluster_sort(dc:pd.DataFrame, cl:int):
    """
    restricts the dataframe dc to cluster value cl
    and returns the results grouped by id, ror sorted
    by the some of probablity descending
    """
    # https://learning.oreilly.com/library/view/streamlit-for-data/9781803248226/text/ch004.xhtml
    dg = dc[dc['paper_cluster'] == cl].copy()
    print(cl)
    dv = dg.groupby(['id','display_name','country_code',
                     'type','r','g','b'])['paper_cluster_score'].sum().to_frame()
    dv.sort_values('paper_cluster_score', ascending=False, inplace=True)
    dv.reset_index(inplace=True) # map the display_name column with the geo_dict to get lattitude, longitude
    dv['latitude'] = dv['display_name'].apply(lambda x: affil_geo_dict.get(x, (None, None))[0])
    dv['longitude'] = dv['display_name'].apply(lambda x: affil_geo_dict.get(x, (None, None))[1])
    kw = centroids[centroids.cluster == cl]['keywords'].iloc[0]
    return dv, kw

In [108]:
def get_country_collaborations_sort(dc:pd.DataFrame, cl:int):
    """
    resticts the dataframe dc to cluster value cl
    and returns the results of paper_id s where there is 
    more than one country_code
    """
    dg = dc[dc['paper_cluster'] == cl].copy()
    dv = dg.groupby('paper_id')['country_code'].apply(lambda x: len(set(x.values))).to_frame()
    dc = dg.groupby('paper_id')['country_code'].apply(lambda x: list(set(x.values))).to_frame()
    dc.columns = ['collab_countries']
    dv.columns = ['country_count']
    dv['collab_countries'] = dc['collab_countries']
    dv.sort_values('country_count',ascending=False, inplace=True)
    di = dfinfo.loc[dv.index].copy()
    di['country_count'] = dv['country_count']
    di['collab_countries'] = dv['collab_countries']
    return di[di['country_count'] > 1]


redo the current pydeck chart to be a 3d column layer. then add in arcs showing the collaborationgs betweeen institutions.   all within that cluster. ok. 

In [109]:
dftriple.head()

,id,display_name,ror,country_code,type,lineage,paper_id,paper_raw_affiliation_strings,paper_author_position,paper_doi,...,paper_author_id,paper_author_display_name,paper_author_orcid,source,source_type,funder_list,fill_color,r,g,b
0,https://openalex.org/I92403157,University of Science and Technology Beijing,https://ror.org/02egmk993,CN,education,[https://openalex.org/I92403157],https://openalex.org/W2740675802,"[Ministry of Education, Key Laboratory of Know...",first,https://doi.org/10.1109/tcyb.2017.2711961,...,https://openalex.org/A5022113595,Wei He,https://orcid.org/0000-0002-8944-9861,IEEE transactions on cybernetics,journal,"[Royal Society, National Key Research and Deve...","[228, 26, 28]",228,26,28
1,https://openalex.org/I92403157,University of Science and Technology Beijing,https://ror.org/02egmk993,CN,education,[https://openalex.org/I92403157],https://openalex.org/W2740675802,"[Ministry of Education, Key Laboratory of Know...",middle,https://doi.org/10.1109/tcyb.2017.2711961,...,https://openalex.org/A5055142075,Haifeng Huang,https://orcid.org/0000-0002-1615-3779,IEEE transactions on cybernetics,journal,"[Royal Society, National Key Research and Deve...","[228, 26, 28]",228,26,28
2,https://openalex.org/I165932596,National University of Singapore,https://ror.org/01tgyzw49,SG,education,[https://openalex.org/I165932596],https://openalex.org/W2740675802,[Department of Electrical and Computer Enginee...,last,https://doi.org/10.1109/tcyb.2017.2711961,...,https://openalex.org/A5069702292,Shuzhi Sam Ge,None,IEEE transactions on cybernetics,journal,"[Royal Society, National Key Research and Deve...","[228, 26, 28]",228,26,28
3,https://openalex.org/I76569877,Southeast University,https://ror.org/04ct4d772,CN,education,[https://openalex.org/I76569877],https://openalex.org/W2418767125,"[School of Automation, Southeast University, N...",first,https://doi.org/10.1109/tsmc.2016.2562506,...,https://openalex.org/A5019248683,Changyin Sun,https://orcid.org/0000-0001-9269-334X,"IEEE transactions on systems, man, and cyberne...",journal,[National Natural Science Foundation of China],"[228, 26, 28]",228,26,28
4,https://openalex.org/I92403157,University of Science and Technology Beijing,https://ror.org/02egmk993,CN,education,[https://openalex.org/I92403157],https://openalex.org/W2418767125,[School of Automation and Electrical Engineeri...,middle,https://doi.org/10.1109/tsmc.2016.2562506,...,https://openalex.org/A5061536058,Wei He,https://orcid.org/0000-0001-8321-8256,"IEEE transactions on systems, man, and cyberne...",journal,[National Natural Science Foundation of China],"[228, 26, 28]",228,26,28


In [110]:
dftriple.columns

Index(['id', 'display_name', 'ror', 'country_code', 'type', 'lineage',
       'paper_id', 'paper_raw_affiliation_strings', 'paper_author_position',
       'paper_doi', 'paper_title', 'paper_abstract', 'paper_publication_date',
       'paper_publication_year', 'paper_grants', 'paper_locations',
       'paper_is_corrresponding', 'paper_x', 'paper_y', 'paper_cluster',
       'paper_cluster_score', 'paper_author_id', 'paper_author_display_name',
       'paper_author_orcid', 'source', 'source_type', 'funder_list',
       'fill_color', 'r', 'g', 'b'],
      dtype='object')

In [111]:
dftriple['paper_cluster'].value_counts().head(15)

-1     6062
 1     6000
 11    4022
 61    2003
 2     1510
 39    1503
 58    1188
 63    1066
 41    1004
 6      933
 7      580
 56     551
 33     520
 25     506
 22     481
Name: paper_cluster, dtype: int64

In [112]:
selected_cluster = 22

In [113]:
dvaffils, kwwaffils = get_affils_cluster_sort(dftriple, selected_cluster)

22


In [114]:
dg = dvaffils.copy()
dg = dg.dropna(subset=["longitude","latitude"])
dg['size'] = 100*dg['paper_cluster_score']
dg = pd.read_json(dg.to_json())

mean_lat = dg['latitude'].mean()

In [115]:
mean_lon = dg['longitude'].mean()

In [116]:
dg.head()

,id,display_name,country_code,type,r,g,b,paper_cluster_score,latitude,longitude,size
0,https://openalex.org/I170215575,National University of Defense Technology,CN,education,228,26,28,66.919189,28.19874,112.97087,6691.918890
1,https://openalex.org/I149594827,Xidian University,CN,education,228,26,28,42.384154,34.25833,108.92861,4238.415351
2,https://openalex.org/I36399199,Nanjing University of Science and Technology,CN,education,228,26,28,33.523557,32.03330,118.85192,3352.355722
3,https://openalex.org/I150229711,University of Electronic Science and Technolog...,CN,education,228,26,28,21.559737,30.66667,104.06667,2155.973679
4,https://openalex.org/I82880672,Beihang University,CN,education,228,26,28,15.716794,39.90750,116.39723,1571.679401


make the color depend on the **type**. ok. 

In [117]:
view = pdk.data_utils.compute_view(dg[["longitude", "latitude"]])
view.pitch = 75
view.bearing = 60

In [118]:
affil_layer = pdk.Layer(
    "ColumnLayer",
    data = dg,
    get_position=["longitude","latitude"],
    get_elevation="paper_cluster_score",
    elevation_scale = 2,
    radius_scale = 75,
    radius_min_pixels=5,
    radius_max_pixels=300,
    line_width_min_pixels=1,
    get_radius="size",
   # radius = 20,
    get_fill_color=[180, 0, 200, 140],
    auto_highlight=True,
    pickable=True,
)

In [119]:
#MAPBOX_TOKEN

In [120]:
r = pdk.Deck(
    layers=[affil_layer],
   # api_keys = {'mapbox': MAPBOX_TOKEN},
   # map_provider='mapbox',
   # map_style="mapbox:styles/mapbox/satellite-streets-v11",
    map_style='dark',
    tooltip=True,
    initial_view_state=view
)
r.title = f'{selected_cluster}'
r.to_html(f'{selected_cluster}.html', notebook_display=False, open_browser=False)

need to color the needle by "type"

In [121]:
dftriple['type'].value_counts(dropna=False)

education     25340
facility       5823
government     1930
company        1543
nonprofit       308
other           173
healthcare      147
archive          20
Name: type, dtype: int64

https://colorbrewer2.org/#type=qualitative&scheme=Set1&n=8

8 different qualitative rgb colors

In [122]:
color_education = [228,26,28]
color_facility = [55, 126, 184]
color_government = [77, 175, 74]
color_company = [152, 78, 163]
color_nonprofit = [255, 127, 0]
color_other = [255, 255, 51]
color_healthcare = [166, 86, 40]
color_archive = [247, 129, 191]

In [123]:
fill_color_dict = {
    'education': color_education,
    'facility': color_facility,
    'government': color_government,
    'company': color_company,
    'nonprofit': color_nonprofit,
    'other': color_other,
    'healthcare': color_healthcare,
    'archive': color_archive
}

In [124]:
dftriple['fill_color'] = dftriple['type'].map(fill_color_dict)

In [125]:
dvaffils, kwwaffils = get_affils_cluster_sort(dftriple, selected_cluster)

22


In [126]:
#dg['type'].value_counts()
dftriple[['type','fill_color']].head()

,type,fill_color
0,education,"[228, 26, 28]"
1,education,"[228, 26, 28]"
2,education,"[228, 26, 28]"
3,education,"[228, 26, 28]"
4,education,"[228, 26, 28]"


In [127]:
dftriple['r'] = dftriple['fill_color'].apply(lambda x: x[0])
dftriple['g'] = dftriple['fill_color'].apply(lambda x: x[1])
dftriple['b'] = dftriple['fill_color'].apply(lambda x: x[2])

In [128]:
dvaffils, kwwaffils = get_affils_cluster_sort(dftriple, selected_cluster)

22


In [130]:
dvaffils[['type','r','g','b']].head()

,type,r,g,b
0,education,228,26,28
1,education,228,26,28
2,education,228,26,28
3,education,228,26,28
4,education,228,26,28


need to add in an arclayer showing the collaboration links

In [131]:
dg = dvaffils.copy()
dg = dg.dropna(subset=["longitude","latitude"])
dg['size'] = 100*dg['paper_cluster_score']
dg = pd.read_json(dg.to_json())

mean_lat = dg['latitude'].mean()
#st.write(dg.head())
mean_lon = dg['longitude'].mean()
cl_initial_view = pdk.ViewState(
        latitude = dg['latitude'].mean(),
        longitude = dg['longitude'].mean(),
        zoom = 3
    )
sp_layer = pdk.Layer(
        'ScatterplotLayer',
        data = dg,
        get_position = ['longitude','latitude'],
        radius_scale = 75,
        radius_min_pixels=5,
        radius_max_pixels=300,
        line_width_min_pixels=1,
       # get_radius = 300,
        get_radius = "size",
        pickable=True,
        opacity = 0.4,
      #  get_fill_color = ['paper_cluster_score <= 1 ? 255 ? 
        get_fill_color = [65, 182, 196]
    )

affil_layer = pdk.Layer(
    "ColumnLayer",
    data = dg,
    get_position=["longitude","latitude"],
 #   get_elevation="paper_cluster_score",
    get_elevation="size",
    elevation_scale = 200,
   # radius_scale = 75,
   # radius_min_pixels=5,
   # radius_max_pixels=300,
    line_width_min_pixels=1,
    radius=3000,
   # radius = 20,
  #  get_fill_color=[180, 0, 200, 140],
    get_fill_color = ['r','g','b'],
   # get_fill_color = "fill_color",
    auto_highlight=True,
    pickable=True,
)

In [132]:
r = pdk.Deck(
    layers=[sp_layer,affil_layer],
   # layers=[affil_layer],
    api_keys = {'mapbox': MAPBOX_TOKEN},
    map_provider='mapbox',
   # map_style="mapbox:styles/mapbox/satellite-streets-v11",
    map_style="mapbox://styles/mapbox/light-v10",
   #  map_style="mapbox://styles/mapbox/satellite-streets-v11",
   # map_style='dark',
    tooltip=True,
    initial_view_state=view
)
r.title = f'{selected_cluster}'
r.to_html(f'{selected_cluster}.html', notebook_display=False, open_browser=False)